In [27]:
import pandas as pd
import numpy as np
import ast
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from xgboost import XGBRegressor

In [28]:
df = pd.read_csv("Retail_Transactions_Dataset.csv")

In [29]:
print(df.head())
print(df.info())

   Transaction_ID                 Date      Customer_Name  \
0      1000000000  2022-01-21 06:27:29       Stacey Price   
1      1000000001  2023-03-01 13:01:21   Michelle Carlson   
2      1000000002  2024-03-21 15:37:04        Lisa Graves   
3      1000000003  2020-10-31 09:59:47  Mrs. Patricia May   
4      1000000004  2020-12-10 00:59:59     Susan Mitchell   

                                             Product  Total_Items  Total_Cost  \
0        ['Ketchup', 'Shaving Cream', 'Light Bulbs']            3       71.65   
1  ['Ice Cream', 'Milk', 'Olive Oil', 'Bread', 'P...            2       25.93   
2                                        ['Spinach']            6       41.49   
3                             ['Tissues', 'Mustard']            1       39.34   
4                                      ['Dish Soap']           10       16.42   

   Payment_Method           City        Store_Type  Discount_Applied  \
0  Mobile Payment    Los Angeles    Warehouse Club              True   
1 

In [30]:
df['Product'] = df['Product'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
df['Product'] = df['Product'].apply(lambda x: x[0] if isinstance(x, list) and len(x) > 0 else None)
df['Product'] = df['Product'].astype(str).str.strip()

In [31]:
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')

if df['Date'].isnull().all():
    print("⚠️ Date unusable → creating synthetic Month")
    df['Month'] = np.tile(np.arange(1, 13), len(df)//12 + 1)[:len(df)]
else:
    df['Month'] = df['Date'].dt.month

In [32]:
df['Discount_Applied'] = df['Discount_Applied'].astype(int)


In [33]:
df_grouped = df.groupby(['Product', 'Month']).agg({
    'Total_Items': 'sum',
    'Total_Cost': 'mean',
    'Discount_Applied': 'mean'
}).reset_index()


In [34]:
freq = df_grouped['Product'].value_counts(normalize=True)
df_grouped['Product_freq'] = df_grouped['Product'].map(freq)

In [35]:
y = df_grouped['Total_Items']
X = df_grouped.drop(columns=['Total_Items', 'Product'])

In [36]:
X_train = X[df_grouped['Month'] <= 9]
X_test  = X[df_grouped['Month'] > 9]

y_train = y[df_grouped['Month'] <= 9]
y_test  = y[df_grouped['Month'] > 9]


In [37]:
lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)

print("\n--- Linear Regression ---")
print("MAE:", mean_absolute_error(y_test, y_pred_lr))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_lr)))



--- Linear Regression ---
MAE: 646.3513808602792
RMSE: 907.7220472753185


In [38]:
xgb = XGBRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=6,
    random_state=42
)

xgb.fit(X_train, y_train)

y_pred_xgb = xgb.predict(X_test)

print("\n--- XGBoost ---")
print("MAE:", mean_absolute_error(y_test, y_pred_xgb))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_xgb)))


--- XGBoost ---
MAE: 298.82171630859375
RMSE: 699.3903595274959


In [39]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [40]:
ann = MLPRegressor(
    hidden_layer_sizes=(32, 16),
    activation='relu',
    max_iter=200,
    early_stopping=True,
    random_state=42
)

In [41]:
ann.fit(X_train_scaled, y_train)

C:\Users\srini\AppData\Roaming\Python\Python312\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


,"loss loss: {'squared_error', 'poisson'}, default='squared_error'The loss function to use when training the weights. Note that the""squared error"" and ""poisson"" losses actually implement""half squares error"" and ""half poisson deviance"" to simplify thecomputation of the gradient. Furthermore, the ""poisson"" loss internally usesa log-link (exponential as the output activation function) and requires``y >= 0``... versionchanged:: 1.7 Added parameter `loss` and option 'poisson'.",'squared_error'
,"hidden_layer_sizes hidden_layer_sizes: array-like of shape(n_layers - 2,), default=(100,)The ith element represents the number of neurons in the ithhidden layer.","(32, ...)"
,"activation activation: {'identity', 'logistic', 'tanh', 'relu'}, default='relu'Activation function for the hidden layer.- 'identity', no-op activation, useful to implement linear bottleneck, returns f(x) = x- 'logistic', the logistic sigmoid function, returns f(x) = 1 / (1 + exp(-x)).- 'tanh', the hyperbolic tan function, returns f(x) = tanh(x).- 'relu', the rectified linear unit function, returns f(x) = max(0, x)",'relu'
,"solver solver: {'lbfgs', 'sgd', 'adam'}, default='adam'The solver for weight optimization.- 'lbfgs' is an optimizer in the family of quasi-Newton methods.- 'sgd' refers to stochastic gradient descent.- 'adam' refers to a stochastic gradient-based optimizer proposed by Kingma, Diederik, and Jimmy BaFor a comparison between Adam optimizer and SGD, see:ref:`sphx_glr_auto_examples_neural_networks_plot_mlp_training_curves.py`.Note: The default solver 'adam' works pretty well on relativelylarge datasets (with thousands of training samples or more) in terms ofboth training time and validation score.For small datasets, however, 'lbfgs' can converge faster and performbetter.",'adam'
,"alpha alpha: float, default=0.0001Strength of the L2 regularization term. The L2 regularization termis divided by the sample size when added to the loss.",0.0001
,"batch_size batch_size: int, default='auto'Size of minibatches for stochastic optimizers.If the solver is 'lbfgs', the regressor will not use minibatch.When set to ""auto"", `batch_size=min(200, n_samples)`.",'auto'
,"learning_rate learning_rate: {'constant', 'invscaling', 'adaptive'}, default='constant'Learning rate schedule for weight updates.- 'constant' is a constant learning rate given by 'learning_rate_init'.- 'invscaling' gradually decreases the learning rate ``learning_rate_`` at each time step 't' using an inverse scaling exponent of 'power_t'. effective_learning_rate = learning_rate_init / pow(t, power_t)- 'adaptive' keeps the learning rate constant to 'learning_rate_init' as long as training loss keeps decreasing. Each time two consecutive epochs fail to decrease training loss by at least tol, or fail to increase validation score by at least tol if 'early_stopping' is on, the current learning rate is divided by 5.Only used when solver='sgd'.",'constant'
,"learning_rate_init learning_rate_init: float, default=0.001The initial learning rate used. It controls the step-sizein updating the weights. Only used when solver='sgd' or 'adam'.",0.001
,"power_t power_t: float, default=0.5The exponent for inverse scaling learning rate.It is used in updating effective learning rate when the learning_rateis set to 'invscaling'. Only used when solver='sgd'.",0.5
,"max_iter max_iter: int, default=200Maximum number of iterations. The solver iterates until convergence(determined by 'tol') or this number of iterations. For stochasticsolvers ('sgd', 'adam'), note that this determines the number of epochs(how many times each data point will be used), not the number ofgradient steps.",200
,"shuffle shuffle: bool, default=TrueWhether to shuffle samples in each iteration. Only used whensolver='sgd' or 'adam'.",True


In [42]:
y_pred_ann = ann.predict(X_test_scaled)

In [43]:
from sklearn.metrics import r2_score

print("\n--- ANN ---")
print("MAE:", mean_absolute_error(y_test, y_pred_ann))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_ann)))



--- ANN ---
MAE: 4582.591515588791
RMSE: 4628.975109123848


In [44]:
print("\n=== FINAL COMPARISON ===")

print("LR RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_lr)))
print("XGB RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_xgb)))
print("ANN RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_ann)))


=== FINAL COMPARISON ===
LR RMSE: 907.7220472753185
XGB RMSE: 699.3903595274959
ANN RMSE: 4628.975109123848


In [45]:
print("Min:", y.min())
print("Max:", y.max())
print("Mean:", y.mean())

Min: 4389
Max: 13063
Mean: 5654.260288065843


In [46]:
def predict_demand(product, month, discount):

    product = product.strip().lower()

    df_grouped['Product_lower'] = df_grouped['Product'].str.lower()
    filtered = df_grouped[df_grouped['Product_lower'] == product]

    if filtered.empty:
        print("❌ Product not found! Use one from list below.")
        print(df_grouped['Product'].unique()[:10])
        return None

    product_freq = filtered['Product_freq'].iloc[0]

    input_data = {
        'Month': month,
        'Total_Cost': df_grouped['Total_Cost'].mean(),
        'Discount_Applied': discount,
        'Product_freq': product_freq
    }

    input_df = pd.DataFrame([input_data])
    input_df = input_df.reindex(columns=X.columns, fill_value=0)

    return xgb.predict(input_df)[0]

In [47]:
print("\nAvailable Products:")

products = sorted(df_grouped['Product'].unique())
for p in products:
    print(p)

product = input("\nEnter product name (copy from above): ")
month = int(input("Enter month (1-12): "))

discount_input = input("Discount applied? (Yes/No): ").lower()
discount = 1 if discount_input == "yes" else 0

result = predict_demand(product, month, discount)

if result is not None:
    print("\nPredicted Demand:", round(result), "units")


Available Products:
Air Freshener
Apple
BBQ Sauce
Baby Wipes
Banana
Bath Towels
Beef
Bread
Broom
Butter
Canned Soup
Carrots
Cereal
Cereal Bars
Cheese
Chicken
Chips
Cleaning Rags
Cleaning Spray
Coffee
Deodorant
Diapers
Dish Soap
Dishware
Dustpan
Eggs
Extension Cords
Feminine Hygiene Products
Garden Hose
Hair Gel
Hand Sanitizer
Honey
Ice Cream
Insect Repellent
Iron
Ironing Board
Jam
Ketchup
Laundry Detergent
Lawn Mower
Light Bulbs
Mayonnaise
Milk
Mop
Mustard
Olive Oil
Onions
Orange
Pancake Mix
Paper Towels
Pasta
Peanut Butter
Pickles
Plant Fertilizer
Potatoes
Power Strips
Razors
Rice
Salmon
Shampoo
Shaving Cream
Shower Gel
Shrimp
Soap
Soda
Spinach
Sponges
Syrup
Tea
Tissues
Toilet Paper
Tomatoes
Toothbrush
Toothpaste
Trash Bags
Trash Cans
Tuna
Vacuum Cleaner
Vinegar
Water
Yogurt

Predicted Demand: 6563 units


In [52]:
def run_trend_analysis():

    print("\nAvailable Products:")
    products = sorted(df_grouped['Product'].unique())

    for p in products:
        print(p)

    product = input("\nEnter product name (copy exactly): ").strip()
    month = int(input("Enter month (1-12): "))

    discount_input = input("Discount applied? (Yes/No): ").lower()
    discount = 1 if discount_input == "yes" else 0

    # ---------- CURRENT DEMAND ----------
    df_grouped['Product_lower'] = df_grouped['Product'].str.lower()

    current_df = df_grouped[
        (df_grouped['Product_lower'] == product.lower()) &
        (df_grouped['Month'] == month)
    ]

    if current_df.empty:
        print("❌ No historical data for this product/month")
        return

    current_demand = current_df['Total_Items'].values[0]

    # ---------- PREDICTED DEMAND ----------
    predicted = predict_demand(product, month, discount)

    if predicted is None:
        return

    predicted = round(predicted)

    # ---------- CHANGE ----------
    change = predicted - current_demand
    percent_change = (change / current_demand) * 100

    # ---------- OUTPUT ----------
    print("\n📊 DEMAND TREND ANALYSIS")
    print("Product:", product)
    print("Month:", month)

    print("\nCurrent Demand:", current_demand)
    print("Predicted Demand:", predicted)

    if change > 0:
        print("\n📈 Demand Increased by:", change, "units")
    elif change < 0:
        print("\n📉 Demand Decreased by:", abs(change), "units")
    else:
        print("\n➡️ Demand Unchanged")

    print("Change (%):", round(percent_change, 2), "%")

In [55]:
run_trend_analysis()


Available Products:
Air Freshener
Apple
BBQ Sauce
Baby Wipes
Banana
Bath Towels
Beef
Bread
Broom
Butter
Canned Soup
Carrots
Cereal
Cereal Bars
Cheese
Chicken
Chips
Cleaning Rags
Cleaning Spray
Coffee
Deodorant
Diapers
Dish Soap
Dishware
Dustpan
Eggs
Extension Cords
Feminine Hygiene Products
Garden Hose
Hair Gel
Hand Sanitizer
Honey
Ice Cream
Insect Repellent
Iron
Ironing Board
Jam
Ketchup
Laundry Detergent
Lawn Mower
Light Bulbs
Mayonnaise
Milk
Mop
Mustard
Olive Oil
Onions
Orange
Pancake Mix
Paper Towels
Pasta
Peanut Butter
Pickles
Plant Fertilizer
Potatoes
Power Strips
Razors
Rice
Salmon
Shampoo
Shaving Cream
Shower Gel
Shrimp
Soap
Soda
Spinach
Sponges
Syrup
Tea
Tissues
Toilet Paper
Tomatoes
Toothbrush
Toothpaste
Trash Bags
Trash Cans
Tuna
Vacuum Cleaner
Vinegar
Water
Yogurt

📊 DEMAND TREND ANALYSIS
Product: Air Freshener
Month: 5

Current Demand: 5949
Predicted Demand: 6048

📈 Demand Increased by: 99 units
Change (%): 1.66 %


In [51]:
import pickle

columns = X.columns

pickle.dump(lr, open("lr.pkl", "wb"))
pickle.dump(xgb, open("xgb.pkl", "wb"))
pickle.dump(ann, open("ann.pkl", "wb"))
pickle.dump(columns, open("columns.pkl", "wb"))
pickle.dump(products, open("products.pkl", "wb"))
pickle.dump(df_grouped, open("df_grouped.pkl", "wb"))